##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini TTS: Voice Design and Voice Replication

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_Started_Voices.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

With **Gemini 3.8 Text-to-Speech (TTS)** (`gemini-3.8-flash-tts` and `gemini-3.8-flash-lite-tts`), you are no longer limited to prebuilt system voices. The `client.voices` API lets you create custom voice identities tailored to your application:

1. **[Voice Design (`type="prompted"`)](#voice_design)**: Generate brand-new synthetic voice personas from scratch using a natural language description of age, accent, vocal timbre, cadence, and character archetype.
2. **[Voice Replication (`type="replicated"`)](#voice_replication)**: Clone a real speaker's voice from a short reference audio recording and a spoken consent statement—using either **stateful storage** (`store=True`, returning a reusable `voice_...` ID) or **stateless encryption** (`store=False`, returning a client-managed `voicekey_...` blob).
3. **[Directing & Combining Designed and Replicated Voices](#design_on_replication)**: Apply Gemini 3.8 TTS style prompts (`speech_metadata`) and inline vocal tags (`<laugh>`, `<sigh>`, `<short pause>`) on top of a replicated voice to transform its delivery, and pair your **Designed Voice** with your **Replicated Voice** in a multi-speaker dialogue.
4. **[Managing Stored Custom Voices (`list`, `get`, `delete`)](#manage_voices)**: List the custom voices stored in your project (`client.voices.list`), inspect voice metadata (`client.voices.get`), and delete custom voices (`client.voices.delete`).

> **Tip:** For core single-speaker and multi-speaker Text-to-Speech fundamentals, check out [Get_started_TTS.ipynb](Get_started_TTS.ipynb).

<a name="setup"></a>
## Setup

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

### Install and initialize the SDK

Install `google-genai>=2.25.0`, which includes the `client.voices` API (`create`, `list`, `get`, `delete`) for Voice Design and Voice Replication, and initialize the client:

In [ ]:
%pip install -U -q "google-genai>=2.24.0"  # 2.24+ for Gemini 3.8 TTS & Voices API

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

### Select a model

Both `gemini-3.8-flash-tts` and `gemini-3.8-flash-lite-tts` support Voice Design (`type="prompted"`) and Voice Replication (`type="replicated"`).

In [ ]:
MODEL_ID = "gemini-3.8-flash-tts"  # @param ["gemini-3.8-flash-tts", "gemini-3.8-flash-lite-tts"] {"allow-input": true, "isTemplate": true}

### Audio playback helper

`client.interactions.create` calls with `response_format={'type': 'audio'}` on Gemini 3.8 TTS return base64-encoded 24kHz mono WAV audio (`interaction.output_audio.data`), matching `CreateVoice` preview samples (`sample_audio.data`). The helper below decodes and plays either format directly in the notebook.

In [ ]:
# @title Helper functions (just run that cell) { display-mode: "form" }
import base64
import io
import wave
from IPython.display import Audio, display


def play_audio(audio_or_interaction, rate: int = 24000):
  """Plays audio from interaction.output_audio, an Interaction, base64 string, or WAV/PCM bytes."""
  if hasattr(audio_or_interaction, "output_audio"):
    raw_bytes = base64.b64decode(audio_or_interaction.output_audio.data)
  elif hasattr(audio_or_interaction, "data"):
    raw_bytes = base64.b64decode(audio_or_interaction.data)
  elif isinstance(audio_or_interaction, str):
    raw_bytes = base64.b64decode(audio_or_interaction)
  else:
    raw_bytes = audio_or_interaction

  if raw_bytes[:4] == b"RIFF":
    display(Audio(raw_bytes, autoplay=False))
  else:
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
      wf.setnchannels(1)
      wf.setsampwidth(2)
      wf.setframerate(rate)
      wf.writeframes(raw_bytes)
    display(Audio(buf.getvalue(), autoplay=False))


def play_response(interaction):
  """Plays interaction.output_audio."""
  play_audio(interaction.output_audio)

<a name="voice_design"></a>
## 1. Voice Design (`type="prompted"`)

**Voice Design** lets you create a brand-new synthetic voice persona from a natural language description (`prompted.input`).

When you call `client.voices.create` with `store=True` and `type="prompted"`:
- The API generates a distinct voice identity matching your prompt and returns a persistent **Voice ID** (`voice_...`) stored for 7 days (`expire_time`).
- It also returns a short **`sample_audio`** preview so you can audition the voice immediately before using it in `client.interactions.create`.

### 1.1 Create a designed voice from a text prompt

In [ ]:
designed_voice = client.voices.create(
    store=True,
    voice={
        "model": MODEL_ID,
        "type": "prompted",
        "display_name": "The Grizzled Detective",
        "prompted": {
            "input": (
                "A world-weary 1940s noir private detective in his late 50s "
                "with a gravelly baritone voice, subtle Mid-Atlantic accent, "
                "and unhurried, deliberate pacing."
            )
        },
    },
)

print(f"Created Designed Voice ID : {designed_voice.id}")
print(f"Display Name              : {designed_voice.display_name}")
print(f"Expires At                : {designed_voice.expire_time}")

if designed_voice.sample_audio and designed_voice.sample_audio.data:
    print("\nAuditioning generated sample preview:")
    play_audio(designed_voice.sample_audio.data)

### 1.2 Synthesize speech with your designed voice

Pass `designed_voice.id` in `speech_config.voice_config.voice` when calling `client.interactions.create`.

In [7]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=(
        "It was raining on Figueroa Street when the phone rang. "
        "<short pause> I knew right away this wasn't going to be an ordinary"
        " case."
    ),
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"voice": designed_voice.id}
        ]
    },
)

play_audio(interaction.output_audio)

### 1.3 Generate multiple candidate takes

Because Voice Design is generative, each call to `client.voices.create` produces a unique voice variation matching your prompt. A common workflow is to generate 2–3 candidate takes, listen to each `sample_audio` preview, and keep your favorite.

In [8]:
for take in range(1, 4):
  candidate = client.voices.create(
      store=True,
      voice={
          "model": MODEL_ID,
          "type": "prompted",
          "display_name": f"Radio Host Take {take}",
          "prompted": {
              "input": (
                  "An energetic late-night jazz radio host with a velvet"
                  " baritone voice and a relaxed, gravelly chuckle."
              )
          },
      },
  )
  print(f"Take {take} preview:")
  preview_res = client.interactions.create(
      model=MODEL_ID,
      input="You're listening to Midnight Blue on 88.5 FM.",
      response_format={"type": "audio"},
      generation_config={
          "speech_config": [
              {"voice": candidate.id}
          ]
      },
  )
  play_audio(preview_res.output_audio)

Take 1 preview:


Take 2 preview:


Take 3 preview:


### 1.4 Best practices for Voice Design prompts

| Dimension | What to specify | Example descriptors |
| :--- | :--- | :--- |
| **Age & Gender** | Apparent age bracket and vocal register | *"young adult male"*, *"woman in her 60s"*, *"mid-30s androgynous voice"* |
| **Accent & Origin** | Regional dialect or pronunciation flavor | *"warm Scottish Highlands accent"*, *"neutral General American"*, *"soft Nigerian English accent"* |
| **Vocal Quality & Timbre** | Physical texture and resonance of the voice | *"deep and gravelly"*, *"velvety contralto"*, *"bright, airy, and resonant"* |
| **Pacing & Cadence** | Baseline rhythm and speech tempo | *"unhurried and meditative"*, *"brisk and articulate"*, *"measured documentary cadence"* |
| **Character Archetype** | Role or persona that grounds the voice | *"1940s noir detective"*, *"cozy audiobook narrator"*, *"late-night jazz radio host"* |

> **Important:** Use `CreateVoice` (`prompted.input`) to define **who the speaker is** (their permanent vocal identity), and use `speech_metadata` at synthesis time to control **how they deliver a specific line** (e.g., whispering, laughing, or sounding surprised).

<a name="voice_replication"></a>
## 2. Voice Replication (`type="replicated"`)

**Voice Replication** clones a real speaker's voice from two audio recordings (`audio/wav`, 24kHz mono 16-bit PCM recommended):

1. **Reference Audio (`source_audio`)**: A clean recording of the speaker's natural voice (no background music, room echo, or synthetic/SynthID watermarks).
2. **Consent Audio (`consent_audio`)**: A recording of the **same speaker** in the **same acoustic environment** reciting the mandatory consent statement verbatim:
   > *"I am the owner of this voice and I consent to Google using this voice to create a synthetic voice model."*

> **Pro Tip — Passing Biometric Verification:**
> The API runs two checks during `client.voices.create(type="replicated")`:
> 1. **Transcript verification** on `consent_audio` to confirm the exact consent statement was spoken cleanly (trim leading/trailing silence or mic clicks so words aren't clipped).
> 2. **Speaker similarity verification** comparing the acoustic embeddings of `source_audio` and `consent_audio`. Always record both files with the **same microphone in the same room** (you can even pass a clean, 6+ second studio consent recording for both `source_audio` and `consent_audio` to guarantee a `1.0` acoustic similarity match).

### 2.1 Record or upload your voice & consent audio

Because the Voices API runs anti-spoofing and synthetic speech detectors (`SynthID` / `C2PA` and acoustic deepfake verification) to ensure only real human voices with explicit speaker consent can be cloned, **AI-generated voices cannot be used as reference or consent audio**.

Run the cell below in Google Colab to either **🎙️ Record with Microphone** (reading the consent statement aloud) or **📂 Upload a `.wav` File**:

In [9]:
# @title 🎙️ Record or Upload your Voice & Consent Audio { display-mode: "form" }
import base64
import os
import subprocess
from IPython.display import Audio, HTML, display

CONSENT_STATEMENT = (
    "I am the owner of this voice and I consent to Google using this voice"
    " to create a synthetic voice model."
)
SAMPLE_PATH = "voice_replication_consent.wav"
sample_audio_b64 = None


def _save_recorded_audio(b64_str):
  """Callback invoked by the browser MediaRecorder widget."""
  global sample_audio_b64
  raw_bytes = base64.b64decode(b64_str.split(",")[1])
  with open("temp_recording.webm", "wb") as f:
    f.write(raw_bytes)
  subprocess.run(
      [
          "ffmpeg", "-y", "-i", "temp_recording.webm",
          "-ar", "24000", "-ac", "1", SAMPLE_PATH,
      ],
      check=True,
      stdout=subprocess.DEVNULL,
      stderr=subprocess.DEVNULL,
  )
  with open(SAMPLE_PATH, "rb") as f:
    wav_bytes = f.read()
  sample_audio_b64 = base64.b64encode(wav_bytes).decode("utf-8")
  print(f"✅ Saved 24kHz mono WAV ({len(wav_bytes):,} bytes) to {SAMPLE_PATH}!")
  display(Audio(wav_bytes, autoplay=False))


def _upload_audio_file():
  """Allows uploading a local .wav file in Google Colab."""
  global sample_audio_b64
  from google.colab import files
  uploaded = files.upload()
  if uploaded:
    fname = next(iter(uploaded.keys()))
    subprocess.run(
        ["ffmpeg", "-y", "-i", fname, "-ar", "24000", "-ac", "1", SAMPLE_PATH],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    with open(SAMPLE_PATH, "rb") as f:
      wav_bytes = f.read()
    sample_audio_b64 = base64.b64encode(wav_bytes).decode("utf-8")
    print(f"✅ Uploaded {fname} ({len(wav_bytes):,} bytes) -> {SAMPLE_PATH}!")
    display(Audio(wav_bytes, autoplay=False))


try:
  from google.colab import output
  output.register_callback("notebook.SaveConsentAudio", _save_recorded_audio)
  output.register_callback("notebook.UploadConsentAudio", _upload_audio_file)
  box_style = (
      "padding:14px; border:1px solid #dadce0; border-radius:8px;"
      " max-width:640px; font-family:sans-serif;"
  )
  quote_style = (
      "margin:8px 0; padding:10px 14px; background:#f8f9fa;"
      " border-left:4px solid #1a73e8; font-style:italic;"
  )
  rec_btn_style = (
      "padding:8px 16px; font-size:14px; border-radius:6px; border:none;"
      " background:#1a73e8; color:white; cursor:pointer; margin-right:8px;"
  )
  up_btn_style = (
      "padding:8px 16px; font-size:14px; border-radius:6px;"
      " border:1px solid #dadce0; background:white; color:#3c4043;"
      " cursor:pointer;"
  )
  display(HTML(f"""
  <div style="{box_style}">
    <div style="font-weight:600; margin-bottom:6px;">
      Read aloud when recording:
    </div>
    <blockquote style="{quote_style}">
      "{CONSENT_STATEMENT}"
    </blockquote>
    <button id="vr-rec-btn" style="{rec_btn_style}">
      🎙️ Record with Microphone
    </button>
    <button id="vr-up-btn" style="{up_btn_style}">
      📂 Upload .wav File
    </button>
    <span id="vr-status" style="margin-left:12px; font-size:13px; color:#5f6368;">
      Ready
    </span>
  </div>
  <script>
  (function() {{
    const btn = document.getElementById('vr-rec-btn');
    const upBtn = document.getElementById('vr-up-btn');
    const status = document.getElementById('vr-status');
    let recorder = null;
    let chunks = [];
    upBtn.onclick = () => {{
      status.textContent = 'Waiting for file upload...';
      google.colab.kernel.invokeFunction(
          'notebook.UploadConsentAudio', [], {{}}
      );
    }};
    btn.onclick = async () => {{
      if (!recorder || recorder.state === 'inactive') {{
        const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
        recorder = new MediaRecorder(stream);
        chunks = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.onstop = () => {{
          const blob = new Blob(chunks, {{ type: 'audio/webm' }});
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          reader.onloadend = () => {{
            status.textContent = 'Converting to 24kHz WAV...';
            google.colab.kernel.invokeFunction(
                'notebook.SaveConsentAudio', [reader.result], {{}}
            );
            status.textContent = '✅ Recording saved!';
          }};
          stream.getTracks().forEach(t => t.stop());
        }};
        recorder.start();
        btn.textContent = '⏹️ Stop Recording';
        btn.style.background = '#d93025';
        status.textContent = '🔴 Recording... Speak the consent phrase now.';
      }} else {{
        recorder.stop();
        btn.textContent = '🎙️ Record Again';
        btn.style.background = '#1a73e8';
      }}
    }};
  }})();
  </script>
  """))
except ImportError:
  print(
      "Running outside Google Colab: place your 24kHz mono"
      f" {SAMPLE_PATH!r} in the working directory."
  )
  if os.path.exists(SAMPLE_PATH):
    with open(SAMPLE_PATH, "rb") as f:
      sample_audio_b64 = base64.b64encode(f.read()).decode("utf-8")

### 2.2 Option A: Stateful Voice Replication (`store=True`)

When `store=True`, the API stores the replicated voice profile on Google servers for 7 days (`expire_time`) and returns a lightweight **`replicated_voice.id`** (`voice_...`) that you can pass to `client.interactions.create`.

In [ ]:
replicated_voice = client.voices.create(
    store=True,
    voice={
        "model": MODEL_ID,
        "type": "replicated",
        "display_name": "My Replicated Voice",
        "replicated": {
            "source_audio": {
                "data": sample_audio_b64,
                "mime_type": "audio/wav",
            },
            "consent_audio": {
                "data": sample_audio_b64,
                "mime_type": "audio/wav",
            },
        },
    },
)

print(f"Stateful Replicated Voice ID : {replicated_voice.id}")
print(f"Expires At                   : {replicated_voice.expire_time}")

replicated_interaction = client.interactions.create(
    model=MODEL_ID,
    input=(
        "Hello! This speech is generated using my stateful "
        "replicated voice ID via the Interactions API."
    ),
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"voice": replicated_voice.id}
        ]
    },
)

play_audio(replicated_interaction.output_audio)

### 2.3 Option B: Stateless Voice Replication (`store=False`)

If you prefer **not** to store the voice profile on Google servers, set `store=False`. Instead of a short `id`, the API returns an encrypted **`stateless_voice.key`** (`voicekey_...`) containing the encoded voice representation. You store this key on your own backend or client device and pass it directly to `voice_config.voice`.

In [ ]:
stateless_voice = client.voices.create(
    store=False,
    voice={
        "model": MODEL_ID,
        "type": "replicated",
        "display_name": "Stateless Replicated Voice",
        "replicated": {
            "source_audio": {
                "data": sample_audio_b64,
                "mime_type": "audio/wav",
            },
            "consent_audio": {
                "data": sample_audio_b64,
                "mime_type": "audio/wav",
            },
        },
    },
)

key_preview = stateless_voice.key[:15]
print(
    f"Stateless Voice Key length: {len(stateless_voice.key):,} chars"
    f" ({key_preview}...)"
)

stateless_interaction = client.interactions.create(
    model=MODEL_ID,
    input=(
        "And this line is synthesized using a stateless encrypted "
        "voice key, with zero server-side voice storage."
    ),
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"voice": stateless_voice.key}
        ]
    },
)

play_audio(stateless_interaction.output_audio)

## 3. Style Direction & Combining Designed and Replicated Voices

How do Voice Design and Voice Replication work together?

While `CreateVoice` uses either `type="prompted"` (to synthesize a new vocal identity from text) or `type="replicated"` (to lock a speaker's biometric vocal timbre from audio), **Gemini 3.8 TTS decouples vocal identity from performance direction**:

1. **Style Direction on a Replicated Voice**: Even if your reference audio was recorded in a calm, neutral reading tone, you can dynamically steer how your replicated voice acts at synthesis time using `speech_metadata={"style": "..."}` and inline vocal tags (`<laugh>`, `<sigh>`, `<chuckle>`, `<short pause>`).
2. **Multi-Speaker Casting (`Designed + Replicated`)**: You can cast your **Designed Voice** (from Section 1) and your **Replicated Voice** (from Section 2) together in the same multi-speaker conversation using `multi_speaker_voice_config`.

### 3.1 Directing performance and style on a Replicated Voice

In [12]:
# Example 1: Steer the replicated voice into a hushed, suspenseful whisper
whisper_interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "text",
            "text": (
                "Wait... did you hear footsteps in the hallway? <short pause> "
                "<sigh> Keep your voice down, they might hear us."
            ),
            "annotations": [
                {
                    "type": "speech_metadata",
                    "style": (
                        "whispering a mysterious secret with tense, hushed"
                        " urgency"
                    ),
                }
            ],
        }
    ],
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"voice": replicated_voice.id}
        ]
    },
)

play_audio(whisper_interaction.output_audio)

Now let's take that exact same `replicated_voice.id` and steer it into an ecstatic, high-energy celebration:

In [13]:
# Example 2: Steer the same replicated voice into an excited celebration
celebration_interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "text",
            "text": (
                "Unbelievable! The deployment went live with zero errors on "
                "the first try! <laugh> What a legendary moment!"
            ),
            "annotations": [
                {
                    "type": "speech_metadata",
                    "style": (
                        "ecstatic, high-energy sports commentator celebrating"
                        " a victory"
                    ),
                }
            ],
        }
    ],
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"voice": replicated_voice.id}
        ]
    },
)

play_audio(celebration_interaction.output_audio)

### 3.2 Multi-speaker dialogue: Pairing your Designed Voice with your Replicated Voice

In `multi_speaker_voice_config`, set `voice_config={"voice": ...}` (rather than `prebuilt_voice_config`) to bind each speaker name to a custom `designed_voice.id` or `replicated_voice.id`, and use `speech_metadata` on each content part to assign the speaker and per-turn delivery style.

In [ ]:
dialogue_interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "text",
            "text": (
                "I have been reviewing the server logs all night. "
                "<short pause> Tell me what really happened in that lab."
            ),
            "annotations": [
                {
                    "type": "speech_metadata",
                    "speaker": "Detective",
                    "style": "gritty, suspicious, and world-weary",
                }
            ],
        },
        {
            "type": "text",
            "text": (
                "We just gave the model a text prompt and a five-second "
                "audio clip! <laugh> Next thing we knew, it was speaking "
                "with both our voices!"
            ),
            "annotations": [
                {
                    "type": "speech_metadata",
                    "speaker": "Creator",
                    "style": "amused, upbeat, and enthusiastic",
                }
            ],
        },
        {
            "type": "text",
            "text": (
                "<sigh> So the synthetic detective and the real engineer "
                "are in the same room now. Case closed."
            ),
            "annotations": [
                {
                    "type": "speech_metadata",
                    "speaker": "Detective",
                    "style": "dryly amused and relaxed",
                }
            ],
        },
    ],
    response_format={"type": "audio"},
    generation_config={
        "speech_config": [
            {"speaker": "Detective", "voice": designed_voice.id},
            {"speaker": "Creator", "voice": replicated_voice.id},
        ]
    },
)

play_audio(dialogue_interaction.output_audio)

<a name="manage_voices"></a>
## 4. Managing Stored Custom Voices (`list`, `get`, `delete`)

When you create custom voices with `store=True` (either via **Voice Design** or **Voice Replication**), they are stored in your project for up to 7 days (`expire_time`). You can list, inspect, and delete your stored custom voices using `client.voices`:

### 4.1 List your custom stored voices

In [15]:
# List custom voices stored in your project
custom_resp = client.voices.list(type_=["prompted", "replicated"])
stored_voices = custom_resp.voices or []
print(f"Stored Custom Voices ({len(stored_voices)} found):")
for v in stored_voices:
  print(
      f"  - {v.id} [{v.type}] : {v.display_name} (expires {v.expire_time})"
  )

Stored Custom Voices (2 found):
  - voice_fake12345678 [prompted] : Noir Detective Narrator (expires 2026-09-30T15:55:00Z)
  - voice_fake87654321 [replicated] : My Replicated Voice (expires 2026-09-30T15:57:00Z)


### 4.2 Inspect and delete stored voices

Use `client.voices.get(id=...)` to inspect a stored voice and `client.voices.delete(id=...)` to remove custom voices when you no longer need them.

In [16]:
# Inspect metadata for our designed voice
fetched_voice = client.voices.get(id=designed_voice.id)
print(
    f"Fetched Voice: {fetched_voice.id} "
    f"({fetched_voice.display_name}) - Type: {fetched_voice.type}"
)

# Clean up the custom voices created in this notebook
client.voices.delete(id=designed_voice.id)
client.voices.delete(id=replicated_voice.id)
print("Deleted custom voices successfully.")

Fetched Voice: voice_fake12345678 (Noir Detective Narrator) - Type: prompted
Deleted custom voices successfully.


## What's Next

- **[Get Started with Text-to-Speech](Get_started_TTS.ipynb)**: Explore single-speaker and multi-speaker speech generation, inline vocal tags (`<laugh>`, `<sigh>`, `<short pause>`), backchannels (`|uh-huh|`), and audio streaming.
- **[Live API Quickstart](Get_started_LiveAPI.ipynb)**: Build real-time, low-latency bidirectional voice conversations with Gemini.
- **Official Documentation**:
  - [Speech Generation Guide](https://ai.google.dev/gemini-api/docs/speech-generation)
  - [Voice Design Guide](https://ai.google.dev/gemini-api/docs/voice-design)
  - [Voice Replication Guide](https://ai.google.dev/gemini-api/docs/voice-replication)